In [ ]:
import numpy as np, math

# ---------- su(3) basis: e_a = i * lambda_a ----------
def gell_mann():
    i = 1j
    lam1 = np.array([[0,1,0],[1,0,0],[0,0,0]],dtype=complex)
    lam2 = np.array([[0,-i,0],[i,0,0],[0,0,0]],dtype=complex)
    lam3 = np.array([[1,0,0],[0,-1,0],[0,0,0]],dtype=complex)
    lam4 = np.array([[0,0,1],[0,0,0],[1,0,0]],dtype=complex)
    lam5 = np.array([[0,0,-i],[0,0,0],[i,0,0]],dtype=complex)
    lam6 = np.array([[0,0,0],[0,0,1],[0,1,0]],dtype=complex)
    lam7 = np.array([[0,0,0],[0,0,-i],[0,i,0]],dtype=complex)
    lam8 = (1/math.sqrt(3))*np.array([[1,0,0],[0,1,0],[0,0,-2]],dtype=complex)
    return [lam1,lam2,lam3,lam4,lam5,lam6,lam7,lam8]

lam = gell_mann()
basis = [1j*L for L in lam]  # orthonormal for <X,Y>=-(1/2)ReTr(XY)

def inner(A,B):
    return -0.5*np.real(np.trace(A@B))

I3 = np.eye(3, dtype=complex)

# exp on SU(3) via diagonalizing Hermitian H = iA  (A anti-Hermitian)
def exp_su3(A):
    H = 1j * A
    w, V = np.linalg.eigh(H)
    U = V @ np.diag(np.exp(-1j*w)) @ V.conj().T
    # enforce det=1
    U = U / np.linalg.det(U)**(1/3)
    return U

def Ad_matrix(U):
    A = np.zeros((8,8), dtype=float)
    for b in range(8):
        Y = U @ basis[b] @ U.conj().T
        for a in range(8):
            A[a,b] = inner(basis[a], Y)
    return A

# ---------- one-plaquette Wilson term ----------
def z_fund(U):
    return 1.0 - (1.0/3.0)*np.real(np.trace(U))

def plaquette_holonomy(Ulinks):
    U0,U1,U2,U3 = Ulinks
    return U0 @ U1 @ U2.conj().T @ U3.conj().T

def S_wilson(Ulinks, beta=1.0):
    return beta * z_fund(plaquette_holonomy(Ulinks))

# ---------- intrinsic Hessian (geodesic finite differences) ----------
def perturb_one(Ulinks, e, gen, h, sgn):
    new = list(Ulinks)
    new[e] = exp_su3(sgn*h*gen) @ new[e]
    return new

def perturb_two_same_edge(Ulinks, e, gen1, gen2, h, s1, s2, order="12"):
    new = list(Ulinks)
    if order == "12":
        new[e] = exp_su3(s1*h*gen1) @ exp_su3(s2*h*gen2) @ new[e]
    else:
        new[e] = exp_su3(s2*h*gen2) @ exp_su3(s1*h*gen1) @ new[e]
    return new

def hessian_geodesic(f, Ulinks, h):
    E = len(Ulinks)
    n = 8*E
    H = np.zeros((n,n), dtype=float)
    f0 = f(Ulinks)

    # diagonals
    for e in range(E):
        for a in range(8):
            i = 8*e + a
            fp = f(perturb_one(Ulinks, e, basis[a], h, +1))
            fm = f(perturb_one(Ulinks, e, basis[a], h, -1))
            H[i,i] = (fp - 2*f0 + fm)/(h*h)

    # off-diagonals
    for i in range(n):
        e_i, a_i = divmod(i,8)
        Xi = basis[a_i]
        for j in range(i+1,n):
            e_j, a_j = divmod(j,8)
            Xj = basis[a_j]

            if e_i != e_j:
                fpp = f(perturb_one(perturb_one(Ulinks,e_i,Xi,h,+1), e_j,Xj,h,+1))
                fpm = f(perturb_one(perturb_one(Ulinks,e_i,Xi,h,+1), e_j,Xj,h,-1))
                fmp = f(perturb_one(perturb_one(Ulinks,e_i,Xi,h,-1), e_j,Xj,h,+1))
                fmm = f(perturb_one(perturb_one(Ulinks,e_i,Xi,h,-1), e_j,Xj,h,-1))
                Hij = (fpp - fpm - fmp + fmm)/(4*h*h)
            else:
                # symmetrize order for same-edge mixed derivatives
                if a_i == a_j:
                    Hij = H[i,i]
                else:
                    # order i then j
                    fpp_12 = f(perturb_two_same_edge(Ulinks,e_i,Xi,Xj,h,+1,+1,"12"))
                    fpm_12 = f(perturb_two_same_edge(Ulinks,e_i,Xi,Xj,h,+1,-1,"12"))
                    fmp_12 = f(perturb_two_same_edge(Ulinks,e_i,Xi,Xj,h,-1,+1,"12"))
                    fmm_12 = f(perturb_two_same_edge(Ulinks,e_i,Xi,Xj,h,-1,-1,"12"))
                    Dij = (fpp_12 - fpm_12 - fmp_12 + fmm_12)/(4*h*h)

                    # order j then i
                    fpp_21 = f(perturb_two_same_edge(Ulinks,e_i,Xi,Xj,h,+1,+1,"21"))
                    fpm_21 = f(perturb_two_same_edge(Ulinks,e_i,Xi,Xj,h,+1,-1,"21"))
                    fmp_21 = f(perturb_two_same_edge(Ulinks,e_i,Xi,Xj,h,-1,+1,"21"))
                    fmm_21 = f(perturb_two_same_edge(Ulinks,e_i,Xi,Xj,h,-1,-1,"21"))
                    Dji = (fpp_21 - fpm_21 - fmp_21 + fmm_21)/(4*h*h)

                    Hij = 0.5*(Dij + Dji)

            H[i,j] = H[j,i] = Hij
    return H

# ---------- gauge orbit map D_U and H/V bases ----------
# vertices: 0(root),1,2,3 ; pin vertex 0
verts_nonroot = [1,2,3]
vix = {v:i for i,v in enumerate(verts_nonroot)}

# link orientations for the plaquette links U0,U1,U2,U3:
# U0: 0->1 ; U1: 1->2 ; U2: 3->2 ; U3: 0->3
edges = [(0,1),(1,2),(3,2),(0,3)]

def D_matrix(Ulinks):
    E = len(Ulinks)
    D = np.zeros((8*E, 8*len(verts_nonroot)), dtype=float)
    for e,(t,h) in enumerate(edges):
        if t != 0:
            it = vix[t]
            D[8*e:8*e+8, 8*it:8*it+8] += np.eye(8)
        if h != 0:
            ih = vix[h]
            D[8*e:8*e+8, 8*ih:8*ih+8] += -Ad_matrix(Ulinks[e])
    return D

def HV_bases(Ulinks):
    D = D_matrix(Ulinks)
    # complete QR gives a full orthonormal basis of R^{8E}
    Q, _ = np.linalg.qr(D, mode='complete')
    s = np.linalg.svd(D, compute_uv=False)
    r = int(np.sum(s > 1e-10))
    Qv = Q[:, :r]   # vertical
    Qh = Q[:, r:]   # horizontal
    return Qh, Qv

# ---------- compute (kappa_H, kappa_V, eta) with Ric = 3I ----------
def constants_BE(Ulinks, beta=1.0, h=1e-4):
    f = lambda U: S_wilson(U, beta=beta)
    Hess = hessian_geodesic(f, Ulinks, h)
    n = Hess.shape[0]
    Ric = 3.0*np.eye(n)          # SU(3) Haar mass in this normalization
    BE  = Hess + Ric

    Qh, Qv = HV_bases(Ulinks)
    BHH = Qh.T @ BE @ Qh
    BVV = Qv.T @ BE @ Qv
    BHV = Qh.T @ BE @ Qv

    kappa_H = float(np.min(np.linalg.eigvalsh(0.5*(BHH+BHH.T))))
    kappa_V = float(np.min(np.linalg.eigvalsh(0.5*(BVV+BVV.T))))
    eta     = float(np.linalg.svd(BHV, compute_uv=False)[0]) if BHV.size else 0.0

    kappa_BE_full  = float(np.min(np.linalg.eigvalsh(0.5*(BE+BE.T))))
    kappa_BE_slice = kappa_H - (eta*eta)/kappa_V

    return kappa_H, kappa_V, eta, kappa_BE_full, kappa_BE_slice

# ---------- demo at identity ----------
U0 = [I3,I3,I3,I3]
print(constants_BE(U0, beta=1.0, h=1e-4))

(5.666666659711936, 3.0000000111022294, 8.980145127119249e-16, 3.000000011102229, 5.666666659711936)


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
su3_hessian_drift_certificate.py

Numerically certifies SU(3) Cartan-chart constants for a single-cell model:

  (c_H, kappa, eta_star, ell, R0)

where
  - c_H     : Haar Hessian floor constant at the origin in this chart
  - kappa   : min eigenvalue of ∇²U on the SAFE tube  ||θ|| ≤ R0
  - eta_star: Lyapunov/Doob parameter for φ(θ) = (η/2) θᵀ Q θ
  - ell     : verified drift gap ell = min_{shell} Ψ_φ(θ) on R0 ≤ ||θ|| ≤ shell_factor·R0
  - R0      : SAFE tube radius

Conventions
-----------
Potential:
  U(θ) = β V(θ) + U_haar(θ)

Generator (Euclidean chart):
  L = Δ - ∇U·∇      (reversible w.r.t. μ ∝ e^{-U})

Doob transform / ground-state style weight:
  h(θ) = exp(-φ(θ)/2)   (NOTE the minus sign)
  L^φ f := h^{-1} L(h f) = Δ f - ∇(U+φ)·∇ f + Ψ_φ f

with
  Ψ_φ(θ) = 1/4 ||∇φ||² - 1/2 Lφ
         = 1/4 ||∇φ||² - 1/2(Δφ - ∇U·∇φ)

For quadratic φ(θ) = (η/2) θᵀ Q θ, Q symmetric:
  ∇φ = η Q θ,     Δφ = η tr(Q)
  Ψ_φ(θ) = (η²/4) ||Qθ||²  - (η/2) tr(Q)  + (η/2) θᵀ Q ∇U(θ)

Haar modes
----------
Two common “Haar” potentials appear in the literature depending on coordinates:

(1) haar_mode="exp" (recommended for a true local chart near identity):
    U_haar(θ) = - log Π_{α>0} ( sin(α·θ/2)/(α·θ/2) )^2
    This is smooth at θ=0 and gives a finite Hessian floor.

(2) haar_mode="torus" (Weyl integration / eigenangle density):
    U_haar(θ) = - log Π_{α>0} sin(α·θ/2)^2
    This is singular on the Weyl walls (including θ=0 in alcove variables).

If you are truly working in the Weyl alcove/eigenangles, use "torus" AND
sample only inside the alcove interior (alcove_sampling=True).
If you are doing a local chart near identity in Lie algebra/exponential coords,
use "exp" AND sample in a disk (alcove_sampling=False).

How to use
----------
Edit the block "USER MODEL: V, gradV, hessV" below.

If you only provide V(θ), the script will finite-difference grad/hess.
For stability and speed, providing analytic gradV and hessV is best.

Dependencies: numpy only.
"""

import math
import numpy as np


# ----------------------------
# Root system: SU(3) = A2
# ----------------------------

def su3_positive_roots(normalization: str = "unit") -> np.ndarray:
    """
    Positive roots for A2 in an orthonormal 2D basis.

    Returns array shape (3,2): [α12, α23, α13] = [α1, α2, α1+α2].

    normalization:
      - "unit":  ||α|| = 1
      - "len2":  ||α||^2 = 2  (multiply by sqrt(2))
    """
    sqrt3 = math.sqrt(3.0)
    a1 = np.array([1.0, 0.0])
    a2 = np.array([-0.5, sqrt3 / 2.0])
    a3 = a1 + a2
    roots = np.stack([a1, a2, a3], axis=0)
    if normalization == "len2":
        roots = math.sqrt(2.0) * roots
    elif normalization != "unit":
        raise ValueError("normalization must be 'unit' or 'len2'")
    return roots


def min_eig_sym(A: np.ndarray) -> float:
    """Minimum eigenvalue of a symmetric 2×2 matrix."""
    return float(np.linalg.eigvalsh(A)[0])


# ----------------------------
# Stable trig helpers
# ----------------------------

def _safe_cot(x: float, eps: float = 1e-10) -> float:
    """cot(x) with series stabilization near 0."""
    ax = abs(x)
    if ax < eps:
        # cot x = 1/x - x/3 - x^3/45 - ...
        return 1.0 / x - x / 3.0
    return math.cos(x) / math.sin(x)


def _safe_csc2(x: float, eps: float = 1e-10) -> float:
    """csc^2(x) with series stabilization near 0."""
    ax = abs(x)
    if ax < eps:
        # csc^2 x = 1/x^2 + 1/3 + O(x^2)
        return 1.0 / (x * x) + 1.0 / 3.0
    s = math.sin(x)
    return 1.0 / (s * s)


# ----------------------------
# Haar potential pieces
# ----------------------------

def haar_grad(theta: np.ndarray, roots: np.ndarray, mode: str) -> np.ndarray:
    """
    ∇U_haar for the chosen mode.

    mode="torus":
      U_haar = - log Π sin(α·θ/2)^2
      ∇U_haar = - Σ α cot(α·θ/2)

    mode="exp":
      U_haar = - log Π (sin(α·θ/2)/(α·θ/2))^2
      ∇U_haar = Σ α (1/x - cot x),  x = α·θ/2
    """
    g = np.zeros(2, dtype=float)
    for a in roots:
        x = 0.5 * float(np.dot(a, theta))
        if mode == "torus":
            g += -a * _safe_cot(x)
        elif mode == "exp":
            # 1/x - cot x is smooth at 0: ~ x/3
            ax = abs(x)
            if ax < 1e-10:
                val = x / 3.0
            else:
                val = 1.0 / x - _safe_cot(x)
            g += a * val
        else:
            raise ValueError("haar_mode must be 'torus' or 'exp'")
    return g


def haar_hess(theta: np.ndarray, roots: np.ndarray, mode: str) -> np.ndarray:
    """
    ∇²U_haar for the chosen mode.

    mode="torus":
      ∇²U_haar = Σ (1/2) (α⊗α) csc^2(α·θ/2)

    mode="exp":
      ∇²U_haar = Σ (1/2) (α⊗α) (csc^2 x - 1/x^2), x=α·θ/2
      (finite at 0, with limit (1/6) α⊗α)
    """
    H = np.zeros((2, 2), dtype=float)
    for a in roots:
        x = 0.5 * float(np.dot(a, theta))
        if mode == "torus":
            val = _safe_csc2(x)
        elif mode == "exp":
            ax = abs(x)
            if ax < 1e-10:
                val = 1.0 / 3.0
            else:
                val = _safe_csc2(x) - 1.0 / (x * x)
        else:
            raise ValueError("haar_mode must be 'torus' or 'exp'")
        H += 0.5 * np.outer(a, a) * val
    return H


def haar_floor_matrix(roots: np.ndarray, mode: str) -> np.ndarray:
    """
    The constant Hessian floor at θ=0 for the Haar part in this chart.

    Let S = Σ_{α>0} α⊗α.

    mode="torus": H0 = (1/2) S
    mode="exp"  : H0 = (1/6) S
    """
    S = np.zeros((2, 2), dtype=float)
    for a in roots:
        S += np.outer(a, a)
    if mode == "torus":
        return 0.5 * S
    if mode == "exp":
        return (1.0 / 6.0) * S
    raise ValueError("haar_mode must be 'torus' or 'exp'")


# ----------------------------
# Finite differences (optional)
# ----------------------------

def fd_grad(f, x: np.ndarray, h: float = 1e-6) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    for i in range(x.size):
        e = np.zeros_like(x)
        e[i] = 1.0
        g[i] = (f(x + h * e) - f(x - h * e)) / (2.0 * h)
    return g


def fd_hess(f, x: np.ndarray, h: float = 1e-5) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    n = x.size
    H = np.zeros((n, n), dtype=float)
    fx = f(x)
    for i in range(n):
        ei = np.zeros(n)
        ei[i] = 1.0
        f_ip = f(x + h * ei)
        f_im = f(x - h * ei)
        H[i, i] = (f_ip - 2.0 * fx + f_im) / (h * h)
        for j in range(i + 1, n):
            ej = np.zeros(n)
            ej[j] = 1.0
            f_pp = f(x + h * ei + h * ej)
            f_pm = f(x + h * ei - h * ej)
            f_mp = f(x - h * ei + h * ej)
            f_mm = f(x - h * ei - h * ej)
            Hij = (f_pp - f_pm - f_mp + f_mm) / (4.0 * h * h)
            H[i, j] = Hij
            H[j, i] = Hij
    return H


# ----------------------------
# Sampling utilities
# ----------------------------

def in_su3_alcove(theta: np.ndarray, roots: np.ndarray, delta: float = 1e-6) -> bool:
    """
    Interior test for the SU(3) Weyl alcove in this root basis:
      α1·θ > 0, α2·θ > 0, (α1+α2)·θ < 2π
    """
    a1, a2, a3 = roots
    t1 = float(np.dot(a1, theta))
    t2 = float(np.dot(a2, theta))
    t3 = float(np.dot(a3, theta))
    if t1 <= delta or t2 <= delta:
        return False
    if t3 >= 2.0 * math.pi - delta:
        return False
    return True


def sample_disk(R: float, n_r: int, n_ang: int) -> np.ndarray:
    pts = []
    for r in np.linspace(0.0, R, n_r):
        if r == 0.0:
            continue
        for ang in np.linspace(0.0, 2.0 * math.pi, n_ang, endpoint=False):
            pts.append(np.array([r * math.cos(ang), r * math.sin(ang)], dtype=float))
    return np.array(pts)


def sample_disk_alcove(R: float, n_r: int, n_ang: int, roots: np.ndarray, delta: float) -> np.ndarray:
    pts = []
    for r in np.linspace(0.0, R, n_r):
        if r == 0.0:
            continue
        for ang in np.linspace(0.0, 2.0 * math.pi, n_ang, endpoint=False):
            th = np.array([r * math.cos(ang), r * math.sin(ang)], dtype=float)
            if in_su3_alcove(th, roots, delta=delta):
                pts.append(th)
    return np.array(pts)


def sample_shell(R0: float, R1: float, n_r: int, n_ang: int) -> np.ndarray:
    pts = []
    for r in np.linspace(R0, R1, n_r):
        for ang in np.linspace(0.0, 2.0 * math.pi, n_ang, endpoint=False):
            pts.append(np.array([r * math.cos(ang), r * math.sin(ang)], dtype=float))
    return np.array(pts)


def sample_shell_alcove(R0: float, R1: float, n_r: int, n_ang: int, roots: np.ndarray, delta: float) -> np.ndarray:
    pts = []
    for r in np.linspace(R0, R1, n_r):
        for ang in np.linspace(0.0, 2.0 * math.pi, n_ang, endpoint=False):
            th = np.array([r * math.cos(ang), r * math.sin(ang)], dtype=float)
            if in_su3_alcove(th, roots, delta=delta):
                pts.append(th)
    return np.array(pts)


# ----------------------------
# Core computations
# ----------------------------

def gradU(theta: np.ndarray, beta: float, gradV, roots: np.ndarray, haar_mode: str) -> np.ndarray:
    return beta * gradV(theta) + haar_grad(theta, roots, mode=haar_mode)


def hessU(theta: np.ndarray, beta: float, hessV, roots: np.ndarray, haar_mode: str) -> np.ndarray:
    return beta * hessV(theta) + haar_hess(theta, roots, mode=haar_mode)


def Psi_phi(theta: np.ndarray, beta: float, gradV, roots: np.ndarray, haar_mode: str,
            Q: np.ndarray, eta: float) -> float:
    """
    Ψ_φ(θ) for φ(θ)=(η/2)θᵀQθ, with h=exp(-φ/2).
    """
    th = np.asarray(theta, dtype=float)
    Qth = Q @ th
    gU = gradU(th, beta, gradV, roots, haar_mode=haar_mode)
    return (0.25 * eta * eta * float(Qth @ Qth)
            - 0.5 * eta * float(np.trace(Q))
            + 0.5 * eta * float(th @ Q @ gU))


def certify_kappa(beta: float, roots: np.ndarray, haar_mode: str,
                  V=None, gradV=None, hessV=None,
                  R0: float = 0.5, n_r: int = 40, n_ang: int = 180,
                  fd_step: float = 1e-6,
                  alcove_sampling: bool = False, alcove_delta: float = 1e-6):
    if gradV is None:
        if V is None:
            raise ValueError("Provide gradV or V for finite-difference grad.")
        gradV = lambda x: fd_grad(V, x, h=fd_step)

    if hessV is None:
        if V is None:
            raise ValueError("Provide hessV or V for finite-difference Hessian.")
        hessV = lambda x: fd_hess(V, x, h=fd_step * 10.0)

    pts = sample_disk_alcove(R0, n_r, n_ang, roots, alcove_delta) if alcove_sampling else sample_disk(R0, n_r, n_ang)

    kappa = float("inf")
    kappa_pt = None
    mV = float("inf")
    mV_pt = None

    for th in pts:
        Hv = hessV(th)
        Hu = hessU(th, beta, hessV, roots, haar_mode=haar_mode)
        ev = min_eig_sym(Hv)
        eu = min_eig_sym(Hu)
        if ev < mV:
            mV = ev
            mV_pt = th.copy()
        if eu < kappa:
            kappa = eu
            kappa_pt = th.copy()

    return {
        "kappa": kappa,
        "kappa_argmin": kappa_pt,
        "mV": mV,
        "mV_argmin": mV_pt,
        "num_points": int(pts.shape[0]),
    }


def certify_shell(beta: float, roots: np.ndarray, haar_mode: str,
                  V=None, gradV=None,
                  Q: np.ndarray = None, eta: float = 1.0,
                  R0: float = 0.5, shell_factor: float = 1.5,
                  n_r: int = 30, n_ang: int = 240,
                  fd_step: float = 1e-6,
                  alcove_sampling: bool = False, alcove_delta: float = 1e-6):
    if gradV is None:
        if V is None:
            raise ValueError("Provide gradV or V for finite-difference grad.")
        gradV = lambda x: fd_grad(V, x, h=fd_step)

    if Q is None:
        Q = np.eye(2)

    R1 = shell_factor * R0
    pts = sample_shell_alcove(R0, R1, n_r, n_ang, roots, alcove_delta) if alcove_sampling else sample_shell(R0, R1, n_r, n_ang)

    psi_min = float("inf")
    psi_pt = None
    for th in pts:
        psi = Psi_phi(th, beta, gradV, roots, haar_mode=haar_mode, Q=Q, eta=eta)
        if psi < psi_min:
            psi_min = psi
            psi_pt = th.copy()

    return {
        "ell": psi_min,
        "ell_argmin": psi_pt,
        "num_points": int(pts.shape[0]),
        "R1": R1,
    }


def eta_threshold_from_kappa(kappa: float, Q: np.ndarray, R0: float) -> float:
    eig = np.linalg.eigvalsh(Q)
    lam_min = float(eig[0])
    if lam_min <= 0.0:
        raise ValueError("Q must be positive definite for this threshold formula.")
    trQ = float(np.trace(Q))
    a = 0.25 * (lam_min * lam_min) * (R0 * R0)
    b = 0.5 * (kappa * lam_min * (R0 * R0) - trQ)
    if b >= 0.0:
        return 0.0
    return -b / a


def find_eta_by_shell_search(beta: float, roots: np.ndarray, haar_mode: str,
                             V=None, gradV=None,
                             Q: np.ndarray = None,
                             R0: float = 0.5, shell_factor: float = 1.5,
                             n_r: int = 30, n_ang: int = 240,
                             fd_step: float = 1e-6,
                             alcove_sampling: bool = False, alcove_delta: float = 1e-6,
                             ell_target: float = 0.0,
                             eta0: float = 1.0,
                             max_iter: int = 30):
    if Q is None:
        Q = np.eye(2)

    def ell_of(eta):
        out = certify_shell(beta, roots, haar_mode, V=V, gradV=gradV, Q=Q, eta=eta,
                            R0=R0, shell_factor=shell_factor, n_r=n_r, n_ang=n_ang,
                            fd_step=fd_step, alcove_sampling=alcove_sampling, alcove_delta=alcove_delta)
        return out["ell"], out

    eta_lo = 0.0
    eta_hi = max(eta0, 1e-12)

    ell_hi, out_hi = ell_of(eta_hi)
    it = 0
    while ell_hi < ell_target and it < max_iter:
        eta_lo = eta_hi
        eta_hi *= 2.0
        ell_hi, out_hi = ell_of(eta_hi)
        it += 1

    if ell_hi < ell_target:
        return {
            "eta_star": eta_hi,
            "ell": ell_hi,
            "note": "Hit max_iter without reaching target; increase max_iter or adjust shell.",
            "shell_report": out_hi,
        }

    out_best = out_hi
    for _ in range(max_iter):
        eta_mid = 0.5 * (eta_lo + eta_hi)
        ell_mid, out_mid = ell_of(eta_mid)
        if ell_mid >= ell_target:
            eta_hi = eta_mid
            out_best = out_mid
        else:
            eta_lo = eta_mid

    return {
        "eta_star": eta_hi,
        "ell": out_best["ell"],
        "shell_report": out_best,
        "note": "Bisection complete.",
    }


# ----------------------------
# USER MODEL: V, gradV, hessV
# ----------------------------

def V_example(theta: np.ndarray) -> float:
    theta = np.asarray(theta, dtype=float)
    return 0.5 * float(theta @ theta)   # (1/2)||θ||²


def gradV_example(theta: np.ndarray) -> np.ndarray:
    theta = np.asarray(theta, dtype=float)
    return theta


def hessV_example(theta: np.ndarray) -> np.ndarray:
    return np.eye(2)


# ----------------------------
# Main
# ----------------------------

def main():
    beta = 1.0
    R0 = 0.50

    root_norm = "unit"          # "unit" or "len2"
    haar_mode = "exp"           # "exp" (smooth) or "torus" (singular)

    tube_n_r = 50
    tube_n_ang = 240
    shell_n_r = 40
    shell_n_ang = 300
    shell_factor = 1.5

    alcove_sampling = (haar_mode == "torus")
    alcove_delta = 1e-5

    fd_step = 2e-6

    Q = np.eye(2)

    # Require a numerical safety margin on the shell: Psi >= ell_target
    ell_target = 1e-3

    V = V_example
    gradV = gradV_example
    hessV = hessV_example

    roots = su3_positive_roots(normalization=root_norm)
    H0 = haar_floor_matrix(roots, mode=haar_mode)
    c_H = min_eig_sym(H0)

    print("\n=== SU(3) Haar floor (chart dependent) ===")
    print(f"root_norm      : {root_norm}")
    print(f"haar_mode      : {haar_mode}")
    print("H_Haar,0 =\n", H0)
    print(f"c_H = lambda_min(H_Haar,0) = {c_H:.12g}")

    tube = certify_kappa(beta, roots, haar_mode,
                         V=V, gradV=gradV, hessV=hessV,
                         R0=R0, n_r=tube_n_r, n_ang=tube_n_ang,
                         fd_step=fd_step,
                         alcove_sampling=alcove_sampling, alcove_delta=alcove_delta)

    print("\n=== Local BE curvature on SAFE tube ===")
    print(f"beta           : {beta}")
    print(f"R0             : {R0}")
    print(f"grid points    : {tube['num_points']}")
    print(f"m_V (grid)     : {tube['mV']:.12g}  at {tube['mV_argmin']}")
    print(f"kappa (grid)   : {tube['kappa']:.12g}  at {tube['kappa_argmin']}")

    kappa = tube["kappa"]

    try:
        eta_cf = eta_threshold_from_kappa(kappa, Q, R0)
    except Exception as e:
        eta_cf = None
        print("\n[warn] closed-form eta threshold failed:", str(e))

    print("\n=== Eta selection ===")
    if eta_cf is not None:
        print(f"eta_threshold_from_kappa = {eta_cf:.12g}")
        eta0 = max(1e-6, 1.10 * eta_cf)
    else:
        eta0 = 1.0
    print(f"eta0 (start)   : {eta0:.12g}")
    print(f"ell_target     : {ell_target:.12g}")

    search = find_eta_by_shell_search(beta, roots, haar_mode,
                                      V=V, gradV=gradV, Q=Q,
                                      R0=R0, shell_factor=shell_factor,
                                      n_r=shell_n_r, n_ang=shell_n_ang,
                                      fd_step=fd_step,
                                      alcove_sampling=alcove_sampling, alcove_delta=alcove_delta,
                                      ell_target=ell_target,
                                      eta0=eta0,
                                      max_iter=35)

    eta_star = search["eta_star"]
    shell = search["shell_report"]

    print("\n=== Drift gap on shell ===")
    print(f"shell_factor   : {shell_factor}")
    print(f"R1             : {shell['R1']}")
    print(f"shell points   : {shell['num_points']}")
    print(f"eta_star       : {eta_star:.12g}")
    print(f"ell (min Psi)  : {shell['ell']:.12g}  at {shell['ell_argmin']}")
    print(f"note           : {search['note']}")

    print("\n=== CERTIFICATE ===")
    print(f"(c_H, kappa, eta_star, ell, R0) = ({c_H:.12g}, {kappa:.12g}, {eta_star:.12g}, {shell['ell']:.12g}, {R0})")


if __name__ == "__main__":
    main()


=== SU(3) Haar floor (chart dependent) ===
root_norm      : unit
haar_mode      : exp
H_Haar,0 =
 [[0.25 0.  ]
 [0.   0.25]]
c_H = lambda_min(H_Haar,0) = 0.25

=== Local BE curvature on SAFE tube ===
beta           : 1.0
R0             : 0.5
grid points    : 11760
m_V (grid)     : 1  at [0.01020408 0.        ]
kappa (grid)   : 1.25000031486  at [-0.00026711 -0.01020058]

=== Eta selection ===
eta_threshold_from_kappa = 13.4999993703
eta0 (start)   : 14.8499993073
ell_target     : 0.001

=== Drift gap on shell ===
shell_factor   : 1.5
R1             : 0.75
shell points   : 12000
eta_star       : 13.4996157109
ell (min Psi)  : 0.00100000025437  at [1.41638472e-16 5.00000000e-01]
note           : Bisection complete.

=== CERTIFICATE ===
(c_H, kappa, eta_star, ell, R0) = (0.25, 1.25000031486, 13.4996157109, 0.00100000025437, 0.5)
